# Fraud Compliance Agent Notebook 07 — Leakage and evaluation design

**Fraud Compliance Agent · P0-04 — evaluation protocol acceptance**  
**Status:** Proposal prepared — review pending  
**Structure:** CRISP-DM decision notebook  
**Decision supported:** P0-04 — evaluation protocol acceptance

---

## In plain English

This notebook designs a **fair test for a future fraud model**. It makes sure the model is not allowed to see information from the future, or the answer to a case, while it is being tested. That mistake is called data leakage and can make a weak model look excellent.

Think of it as setting exam rules before anyone studies the paper: we decide which cases belong in training and which must be kept unseen for testing. This notebook defines and checks that protocol; it does not select a model, choose a payment-action threshold, or claim a performance result.

## Table of contents

1. [Business Understanding](#business-understanding)
2. [Data Understanding](#data-understanding)
3. [Data Preparation](#data-preparation)
4. [Modelling](#modelling)
5. [Evaluation](#evaluation)
6. [Deployment](#deployment)
7. [References and review](#references-and-review)

### How to use this notebook

This is a gated CRISP-DM decision notebook. It may document evidence and blockers, but it must not manufacture a corpus, label, feature, threshold, model result, or approval. Raw provider data, identifiers, credentials, customer data, model weights, and hidden reasoning never enter Git or notebook outputs.

<a id="business-understanding"></a>
## 1. Business Understanding

**Question:** Can the selected source support leakage-safe partitions and meaningful score and policy evaluation?

Define the test before selecting a model or threshold.

The outcome is a reviewable proposed decision only. It cannot approve a contract, train or promote a model, or alter a payment route.


In [ ]:
from __future__ import annotations

import json
import subprocess
from datetime import UTC, datetime
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
while REPOSITORY_ROOT != REPOSITORY_ROOT.parent and not (REPOSITORY_ROOT / "docs" / "project-context.md").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

if not (REPOSITORY_ROOT / "docs" / "project-context.md").exists():
    raise RuntimeError("Run this notebook from inside the fraud-compliance-agent repository.")

def git_revision() -> str:
    """Return the current Git revision without failing a gated protocol run.

    Returns:
        Commit hash or an explicit uncommitted-or-unavailable marker.

    Side effects:
        Runs a read-only Git command; no corpus data or secrets are accessed.
    """
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "uncommitted-or-unavailable"

RUN_CONTEXT = {
    "run_at_utc": datetime.now(UTC).isoformat(),
    "git_revision": git_revision(),
    "notebook_status": "draft-not-run",
}
print("Repository:", REPOSITORY_ROOT)
print("Git revision:", RUN_CONTEXT["git_revision"])
print("Safety: do not print secrets, raw provider payloads, identifiers, or model artifacts.")


<a id="data-understanding"></a>
## 2. Data Understanding

Define the prediction-time boundary and distinguish facts available at decision time from late labels, corrections, future events, and post-decision outcomes. The source must support chronological rather than random partitions.

The next cell checks only for the named review inputs. Presence is not acceptance; missing evidence remains an explicit blocker.


In [ ]:
from pathlib import Path

REQUIRED_REVIEW_INPUTS = [
  "docs/proposals/corpus-and-label-feasibility.proposed.json",
  "docs/proposals/enrichment-pipeline.proposed.json",
  "docs/proposals/feature-availability.proposed.json",
  "docs/proposals/leakage-and-evaluation-design.proposed.json"
]
missing = [path for path in REQUIRED_REVIEW_INPUTS if not (REPOSITORY_ROOT / path).exists()]
if missing:
    print("GATED — this notebook has not run because required review inputs are absent:")
    for path in missing:
        print(f"- {path}")
    print("Do not substitute fabricated inputs. Record the blocker in the matching experiment record.")
else:
    print("Required review inputs are present. Continue only after confirming their approval status.")


<a id="data-preparation"></a>
## 3. Data Preparation

Specify disjoint chronological train, calibration/selection, and untouched final-test partitions. Create sanitised edge cases for future events, late labels, duplicates, corrections, and immature negatives without introducing source records.

Use sanitised schema metadata or explicitly permitted aggregate evidence only. Do not repair unavailable fields, backfill labels, resample classes, or infer missing facts.


<a id="modelling"></a>
## 4. Modelling

This phase defines the constraints for later model work; it does not train a model or tune hyperparameters. A model family cannot compensate for leaked features, immature labels, or an invalid split.

No estimator is fitted in this notebook. Any future Logistic Regression, XGBoost, or challenger comparison belongs in Notebook 08 after this notebook's decision and the leakage protocol are accepted.


<a id="evaluation"></a>
## 5. Evaluation

Freeze PR-AUC, ROC-AUC, calibration, precision, recall, false-positive rate, cohort/slice coverage, and confidence-interval approach where feasible. Keep threshold trade-offs separate from score quality; record fraud-loss, false-decline, review-capacity, and review-cost assumptions without selecting a threshold.

The evidence is incomplete until every required input and named review is available. No score, threshold, route, or promotion may be inferred from this draft.


<a id="deployment"></a>
## 6. Deployment

In this CRISP-DM notebook, deployment means placing a sanitised proposal and experiment record into the authorised review process. It does not mean data ingestion, model serving, policy activation, or a runtime change.

The next cell constructs a non-persistent, sanitised report template only.


In [ ]:
from hashlib import sha256

report = {
    "notebook": "07-leakage-and-evaluation-design",
    "status": "proposal-prepared-review-pending",
    "run_context": RUN_CONTEXT,
    "findings": ["A proposed temporal and leakage protocol is available at docs/proposals/leakage-and-evaluation-design.proposed.json."],
    "limitations": ["The protocol is mechanics-only; cutpoints, label maturity, feature eligibility, and release criteria remain unaccepted."],
    "decision_recommendation": "proposed — no approval or promotion is implied",
}

report_bytes = json.dumps(report, sort_keys=True, indent=2).encode("utf-8")
print("Sanitised report template:", REPOSITORY_ROOT / "docs/proposals/leakage-and-evaluation-design.proposed.json")
print("Template digest:", sha256(report_bytes).hexdigest())
print("Do not write the template until it contains only reviewable, sanitised findings.")


<a id="references-and-review"></a>
## 7. References and review

### References

- [Phase 0 notebook plan](notebook-plan.md)
- [Fast-path model technical specification](../docs/proposals/fast-path-fraud-model-technical-spec.md)
- [Experiment record](../docs/experiments/07-leakage-and-evaluation-design.md)

### Findings, limitations, and next gate

- Findings: sanitised Sparkov timing and overlap evidence supports a proposed chronological protocol.
- Limitations: no numeric cutpoints, feature set, label-maturity policy, or action threshold is accepted.
- Recommendation: **proposed** — do not approve a contract, feature, policy, or model from this template.
- Next gate: update the matching experiment record after a sanitised run, then request the named review decision.

### Reviewer checklist

- [ ] Outputs are cleared and contain no secrets, raw provider data, PII, identifiers, or hidden reasoning.
- [ ] Every result is labelled observed, unsupported, indeterminate, unavailable, or proposed as appropriate.
- [ ] The matching experiment record contains revision, inputs, findings, limitations, and artifact digest.
- [ ] No runtime contract, threshold, model promotion, or payment action was inferred.
